# Demo - Bag-of-Words vs Embeddings
**Day 2 - Session 1**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kpassoubady/gen-ai-companion-intuite/blob/main/day2/demos/notebooks/demo-bow-vs-embeddings.ipynb)

**Goal:** Show why bag-of-words misses meaning and how embeddings fix it.

Part 1 builds bag-of-words count vectors with scikit-learn and shows two
problems: word order is lost ("dog bites man" == "man bites dog") and synonyms
are unrelated slots ("car" vs "automobile"). Part 2 embeds the same sentences
with a small sentence-transformers model and ranks them by cosine similarity,
so you can see meaning captured as direction in vector space.

No API key needed. The embedding model downloads once (~80 MB), then caches.

In [1]:
# Scoped install for this demo (safe to re-run; preinstalled on Colab).
%pip install -q numpy scikit-learn sentence-transformers

In [2]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer

DOCS = [
    "dog bites man",
    "man bites dog",
    "I drove my car to work",
    "I drove my automobile to the office",
    "the weather is sunny today",
]


def cosine(a, b):
    """Cosine similarity between two vectors."""
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

## 1. Bag-of-words (scikit-learn CountVectorizer)
Count vectors lose order and treat synonyms as unrelated slots.

In [3]:
def demo_bag_of_words():
    """Count vectors lose order and treat synonyms as unrelated."""
    print("1. Bag-of-words (scikit-learn CountVectorizer)")
    vectorizer = CountVectorizer()
    matrix = vectorizer.fit_transform(DOCS).toarray()
    print(f"  vocabulary: {list(vectorizer.get_feature_names_out())}")

    s = cosine(matrix[0], matrix[1])
    print(f"\n  'dog bites man' vs 'man bites dog' -> cosine {s:.2f}")
    print("  Identical vectors: order is thrown away.")

    s = cosine(matrix[2], matrix[3])
    print(f"\n  'car' sentence vs 'automobile' sentence -> cosine {s:.2f}")
    print("  Low overlap: synonyms are unrelated slots.")


demo_bag_of_words()

1. Bag-of-words (scikit-learn CountVectorizer)
  vocabulary: ['automobile', 'bites', 'car', 'dog', 'drove', 'is', 'man', 'my', 'office', 'sunny', 'the', 'to', 'today', 'weather', 'work']

  'dog bites man' vs 'man bites dog' -> cosine 1.00
  Identical vectors: order is thrown away.

  'car' sentence vs 'automobile' sentence -> cosine 0.55
  Low overlap: synonyms are unrelated slots.


## 2. Embeddings (sentence-transformers all-MiniLM-L6-v2)
Dense embeddings place similar meaning close together, even without shared
words.

In [4]:
def demo_embeddings():
    """Dense embeddings place similar meaning close together."""
    print("2. Embeddings (sentence-transformers all-MiniLM-L6-v2)")
    from sentence_transformers import SentenceTransformer

    model = SentenceTransformer("all-MiniLM-L6-v2")
    vectors = model.encode(DOCS)

    s = cosine(vectors[2], vectors[3])
    print(f"\n  'car' sentence vs 'automobile' sentence -> cosine {s:.2f}")
    print("  High similarity: embeddings capture shared meaning.")

    query = "How do I commute to my job?"
    q_vec = model.encode([query])[0]
    print(f"\n  Similarity search for query: {query!r}")
    scored = sorted(
        ((cosine(q_vec, v), doc) for v, doc in zip(vectors, DOCS)),
        reverse=True,
    )
    for score, doc in scored:
        print(f"    {score:.2f}  {doc}")


demo_embeddings()

2. Embeddings (sentence-transformers all-MiniLM-L6-v2)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

... (6 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


  'car' sentence vs 'automobile' sentence -> cosine 0.82
  High similarity: embeddings capture shared meaning.

  Similarity search for query: 'How do I commute to my job?'
    0.44  I drove my automobile to the office
    0.43  I drove my car to work
    0.02  the weather is sunny today
    -0.08  dog bites man
    -0.09  man bites dog


## Expected output
- The bag-of-words cosine for "dog bites man" vs "man bites dog" is 1.00:
  identical vectors even though the meaning is reversed.
- The bag-of-words cosine for the "car" vs "automobile" sentences is low:
  synonyms share no words.
- The embedding cosine for "car" vs "automobile" is high: meaning is close.
- The commute query ranks the car/automobile sentences above the weather
  sentence, even though it shares no exact words with either.

**Takeaway:** rank by cosine similarity on embeddings, not word overlap.